In [ ]:
# %% [markdown]
# # M5 Decagon Ensemble: Geometric Signal Audit (Signatory)
# **Aim:** Analyze the Log-Signature embeddings to verify 'Path Momentum' capture.
# **Research Focus:** Iterated Integrals, Path Curvature, and Feature Discriminability.

# %%
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Set research-grade plotting style
plt.style.use('seaborn-v0_8-muted')
sns.set_context("paper", font_scale=1.5)

# %% [markdown]
# ### 1. Theoretical Grounding
# The Signature of a path $X: [0, T] \to \mathbb{R}^d$ is a collection of iterated integrals.
# For M5, we represent sales as a 2D path: $(t, \text{Sales}_t)$.
# - **Level 1 Terms:** Capture total volume (displacement).
# - **Level 2 Terms:** Capture the 'Area' under the curve (momentum).
# - **Level 3 Terms:** Capture 'Curvature' (acceleration/deceleration of demand).

# %%
# 2. Loading the Signatory Artifacts
# These were generated in scripts/generate_graphs.py
sig_path = "../data/graphs/path_signatures.pt"
sigs = torch.load(sig_path).numpy() # [30490, Dim]

# Load labels for categorical audit
sales_df = pd.read_csv("../data/raw/sales_train_evaluation.csv")
dept_labels = sales_df['dept_id'].values

print(f"Signature Tensor Shape: {sigs.shape}")
print(f"Dimension per Node: {sigs.shape[1]}")

# %% [markdown]
# ### 2. Distribution of Geometric Coefficients
# We audit the coefficients to ensure no 'Signal Collapse'. If coefficients 
# are near zero, the signature depth is insufficient or the path is too noisy.

# %%
plt.figure(figsize=(15, 6))
sns.boxplot(data=pd.DataFrame(sigs[:, :15]), palette="viridis")
plt.title("Distribution of First 15 Log-Signature Coefficients")
plt.xlabel("Iterated Integral Index")
plt.ylabel("Coefficient Magnitude")
plt.grid(axis='y', alpha=0.3)
plt.show()

# %% [markdown]
# ### 3. Latent Manifold Visualization (PCA)
# If the Log-Signatures are effective, they should naturally cluster 
# by 'Demand Regime'. We use PCA to see if 'FOODS' and 'HOUSEHOLD' 
# signatures are geometrically distinct.

# %%
# Normalize for PCA
scaler = StandardScaler()
sigs_rescaled = scaler.fit_transform(sigs)

pca = PCA(n_components=2)
coords = pca.fit_transform(sigs_rescaled)

plt.figure(figsize=(12, 10))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=pd.factorize(dept_labels)[0], 
            cmap='tab20', s=1, alpha=0.4)
plt.colorbar(scatter, label="Department ID")
plt.title("Log-Signature Manifold: Geometric Separation by Department")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.show()

# %% [markdown]
# ### 4. Signal-to-Noise Ratio (SNR) Audit
# A critical check for the SigGNN: Are these features correlated with 
# future volatility? We check if Level 3 (Curvature) predicts the 
# 28-day standard deviation.

# %%
# Calculate future volatility (standard deviation of the target 28 days)
targets = torch.load("../data/processed/targets_train.pt").numpy()
volatility = np.std(targets, axis=1)

# Pick a Level 3 coefficient (typically index 10-15 in log-sigs)
curvature_coeff = sigs[:, 12] 

correlation = np.corrcoef(curvature_coeff, volatility)[0, 1]
print(f"Correlation (Curvature vs. Future Volatility): {correlation:.4f}")

if abs(correlation) > 0.3:
    print("✅ High Geometric Signal: SigGNN expert will likely dominate the Meta-Blender.")
else:
    print("⚠️ Low Correlation: Consider increasing signature_depth or path-scaling.")